# Calculate Q4 2024's Google Ads ROAS

*Data Analysis in Python · Topic 5.2*

The paid search team wants the fourth quarter of 2024 in one number: Google Ads' ROAS, and which campaigns did best and worst.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that builds the `spend` table: Google Ads' spend for October to December 2024, in topic 5.1.'s standard columns.
2. In the first cell marked ✏️, create the `revenue` table: one row per date and campaign of the `sessions` table, with the columns `date`, `campaign`, `sessions`, `transactions` and `revenue`. `campaign` is the `session_campaign` column, renamed.
3. In the second cell marked ✏️, create:
   - the `joined` table: the `spend` table joined to the `revenue` table on the date and the campaign, keeping every row of the `spend` table
   - `roas`: the quarter's revenue in the `joined` table divided by its spend
4. In the last cell marked ✏️, create:
   - the `by_campaign` table: one row per campaign, with the campaigns as the index, and the columns `spend`, `revenue` and `roas`, sorted by `roas`, highest first. Look at the first row's spend.
   - `worst_campaign`: the name of the campaign with the lowest ROAS
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `revenue` table has 2,551 rows, one per date and campaign. The `joined` table has the 1,008 rows of the `spend` table, with revenue on every row. `roas` is about `2.5078`. The `by_campaign` table has 16 campaigns, sorted by ROAS: `2024_ca_display_blackfriday` first, with 9.59 on $32.91 of spend, and `worst_campaign` is `"2024_us_display_retargeting"`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-10", "2024-11", "2024-12"]
START, END = "2024-10-01", "2024-12-31"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Google Ads spend in topic 5.1's standard columns. Run it, and leave it as it is.
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
spend = google.rename(columns={"campaign_name": "campaign"})
spend["platform"] = "Google Ads"
spend["spend"] = spend["cost_micros"] / 1_000_000
spend = spend[spend["date"].between(START, END)].reset_index(drop=True)
spend = spend[["date", "platform", "campaign", "impressions", "clicks", "spend"]]
spend.shape

## Revenue per campaign and day

In [ ]:
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue.shape

## The join

In [ ]:
joined = spend.merge(revenue, on=["date", "campaign"], how="left")
roas = joined["revenue"].sum() / joined["spend"].sum()
len(joined) == len(spend), joined["revenue"].isna().sum(), roas

## Campaigns

In [ ]:
by_campaign = joined.groupby("campaign")[["spend", "revenue"]].sum()
by_campaign["roas"] = by_campaign["revenue"] / by_campaign["spend"]
by_campaign = by_campaign.sort_values("roas", ascending=False)
worst_campaign = by_campaign.index[-1]
by_campaign

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    spend = found.get("spend")
    if not isinstance(spend, pd.DataFrame) or len(spend) != 1008 or abs(spend["spend"].sum() - 42831.24) > 0.01:
        results.append((False, "The spend table should be Google Ads' 1,008 rows for October to December 2024, $42,831.24 in all. Run the cell that builds it, and don't change it."))
        spend = None

    revenue = found.get("revenue")
    if not isinstance(revenue, pd.DataFrame):
        results.append((False, "revenue isn't a DataFrame yet. Group the sessions table by date and session_campaign with as_index=False, total sessions, transactions and revenue, and rename session_campaign to campaign."))
    elif "session_campaign" in revenue.columns:
        results.append((False, "The revenue table still has a session_campaign column. Rename it to campaign, so the key has the same name on both sides: rename(columns={\"session_campaign\": \"campaign\"})."))
    elif not {"date", "campaign", "revenue"} <= set(revenue.columns):
        results.append((False, f"The revenue table should have date and campaign as columns, with sessions, transactions and revenue. It has {list(revenue.columns)}. Group with as_index=False."))
    elif len(revenue) != 2551:
        results.append((False, f"The revenue table has {len(revenue):,} rows, but the quarter has 2,551 dates and campaigns. Group by both date and session_campaign."))
    elif abs(revenue["revenue"].sum() - 790894.92) > 0.01:
        results.append((False, f"The revenue table's revenue adds up to {revenue['revenue'].sum():,.2f}, but the sessions table has 790,894.92. Total every row of the sessions table."))
    else:
        results.append((True, "The revenue table has the quarter's 2,551 campaign-days, one row each."))

    joined = found.get("joined")
    if not isinstance(joined, pd.DataFrame):
        results.append((False, "joined isn't a DataFrame yet. spend.merge(revenue, on=[\"date\", \"campaign\"], how=\"left\")."))
    elif "date_x" in joined.columns or "campaign_x" in joined.columns:
        key = "campaign" if "date_x" in joined.columns else "date"
        results.append((False, f"The joined table has {len(joined):,} rows and columns ending in _x and _y: it was joined on {key} alone. Join on both: on=[\"date\", \"campaign\"]."))
    elif len(joined) == 2551:
        results.append((False, "The joined table has 2,551 rows: every campaign-day of the revenue table, email and organic included. Keep the spend table's rows: spend.merge(revenue, ..., how=\"left\")."))
    elif len(joined) != 1008:
        results.append((False, f"The joined table has {len(joined):,} rows, but the spend table has 1,008. A left join from the spend table keeps its rows."))
    elif spend is not None and abs(joined["spend"].sum() - spend["spend"].sum()) > 0.01:
        results.append((False, "The joined table's spend doesn't add up to the spend table's $42,831.24."))
    elif "revenue" not in joined.columns or joined["revenue"].isna().any():
        results.append((False, "Some rows of the joined table have no revenue. Every Google Ads campaign-day in this quarter has a match: join on date and campaign, with the campaign names as they are."))
    else:
        results.append((True, "The joined table has the spend table's 1,008 rows and its $42,831.24, with revenue on every row."))

    roas = found.get("roas")
    if isinstance(roas, (pd.DataFrame, pd.Series)):
        results.append((False, f"roas is a {type(roas).__name__}, not a number. Divide two totals: joined[\"revenue\"].sum() / joined[\"spend\"].sum()."))
    elif not pd.api.types.is_number(roas):
        results.append((False, f"roas should be the joined table's revenue over its spend. It is {roas!r}."))
    elif abs(roas - 3.0377) < 0.0001:
        results.append((False, "roas is 3.04, the mean of the 1,008 daily ROASes. The quarter's ROAS is its revenue over its spend: joined[\"revenue\"].sum() / joined[\"spend\"].sum()."))
    elif abs(roas - 0.3988) < 0.0001:
        results.append((False, "roas is 0.40: spend over revenue. ROAS is revenue divided by spend."))
    elif abs(roas - 2.4705) < 0.0001:
        results.append((False, "roas is 2.47: all of these campaigns' revenue in the sessions table, including 18 days of refunds after their spend had stopped. Use the joined table's revenue."))
    elif abs(roas - 2.5078) > 0.0001:
        results.append((False, f"roas should be the joined table's revenue over its spend, about 2.5078. It is {roas!r}."))
    else:
        results.append((True, f"Google Ads' ROAS for Q4 2024 is {roas:.2f}: ${roas:.2f} of revenue for every dollar."))

    by_campaign = found.get("by_campaign")
    worst = found.get("worst_campaign")
    if not isinstance(by_campaign, pd.DataFrame):
        results.append((False, "by_campaign isn't a DataFrame yet. Group the joined table by campaign, total spend and revenue, and add roas after totaling."))
    elif len(by_campaign) != 16 or "2024_us_search_brand" not in by_campaign.index:
        results.append((False, f"The by_campaign table should have the 16 campaigns as its index. It has {len(by_campaign):,} rows. Group by campaign, without as_index=False."))
    elif not {"spend", "revenue", "roas"} <= set(by_campaign.columns):
        results.append((False, f"The by_campaign table should have the columns spend, revenue and roas. It has {list(by_campaign.columns)}."))
    elif not ((by_campaign["roas"] - by_campaign["revenue"] / by_campaign["spend"]).abs() < 1e-9).all():
        results.append((False, "The by_campaign table's roas should be each campaign's total revenue over its total spend, calculated after grouping."))
    elif by_campaign.index[0] != "2024_ca_display_blackfriday" or not by_campaign["roas"].is_monotonic_decreasing:
        results.append((False, f"The by_campaign table should be sorted by roas, highest first. Its first row is {by_campaign.index[0]}."))
    elif worst != "2024_us_display_retargeting":
        results.append((False, f"worst_campaign should be the name of the campaign with the lowest ROAS, the last row of the sorted table: by_campaign.index[-1]. It is {worst!r}."))
    else:
        results.append((True, "The best ROAS, 9.59, is 2024_ca_display_blackfriday on $32.91 of spend, and the worst is 2024_us_display_retargeting, 1.59 on $2,365.89."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue.shape
```

```python
joined = spend.merge(revenue, on=["date", "campaign"], how="left")
roas = joined["revenue"].sum() / joined["spend"].sum()
len(joined) == len(spend), joined["revenue"].isna().sum(), roas
```

```python
by_campaign = joined.groupby("campaign")[["spend", "revenue"]].sum()
by_campaign["roas"] = by_campaign["revenue"] / by_campaign["spend"]
by_campaign = by_campaign.sort_values("roas", ascending=False)
worst_campaign = by_campaign.index[-1]
by_campaign
```

</details>